# Mamba SOH v1.6 — GH-88 ablation (Kaggle GPU)

Fix coverage gap 4°C high-SOH: split rebalance B0047 val→train (A1) + `--balance-bands` loss (A2) + `--jitter`/`--swa` tren top A1 (A3, GH-88 optimize them).
Baseline so sánh: **v1.5 test MAE 1.98% / RMSE 2.38%** (per-band 75-85% bias -4.5..-11%).

**Trước khi chạy:**
1. Settings → Accelerator → **GPU T4 x2** (KHÔNG chọn P100 — PyTorch Kaggle đã bỏ sm_60)
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. Add-ons → Secrets → `GITHUB_TOKEN` = GitHub PAT (repo private)
4. **Push branch GH-88 lên GitHub trước** — Kaggle clone từ remote


## 1 — GPU check


In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print('GPU:', name)
    assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang T4 x2'
else:
    print('WARNING: bat GPU o Settings -> Accelerator -> GPU T4 x2')


## 2 — Clone branch GH-88


In [ ]:
import subprocess, os
BRANCH  = 'feat/GH-88-soh-v16-coverage-gap'   # push branch nay len GitHub truoc
REPO    = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thu public clone:', e)
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,url,REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git','log','-1','--oneline']).decode())


## 3 — Dependencies


In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)


## 4 — Tìm NASA dataset


In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv - + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET:', DATASET)


## 5 — Preprocess (split mới 24/1/1 — GH-88)

Kiểm tra log: `Train: ... (24 batteries)` và có dòng `B0047` trong train, `Val: ['B0046']`.


In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess.py --data-dir "{DATASET}" --output-dir data/processed


## 6 — Run A1: split mới, loss chuẩn


In [ ]:
import os, shutil; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --data-dir data/processed --epochs 100 --log-dir logs/training
# archive run A1 (train.py ghi de cung 1 path giua cac run)
import sys; sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH
os.makedirs('/kaggle/working/runs/A1', exist_ok=True)
for p in [MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH]:
    shutil.copy(p, '/kaggle/working/runs/A1/')
print('A1 archived')


## 7 — Run A2: split mới + `--balance-bands`


In [ ]:
import os, shutil; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --data-dir data/processed --epochs 100 --log-dir logs/training --balance-bands
import sys; sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH
os.makedirs('/kaggle/working/runs/A2', exist_ok=True)
for p in [MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH]:
    shutil.copy(p, '/kaggle/working/runs/A2/')
print('A2 archived')


## 8 — Run A3: A1 + `--jitter`/`--swa` (optimize them, GH-88)

Xay tren config thang A1 (khong `--balance-bands`) + input jitter 0.0075 + Stochastic Weight Averaging tail epochs. Muc tieu: keo MAE xuong gan stretch <1%.


In [ ]:
import os, shutil; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --data-dir data/processed --epochs 100 --log-dir logs/training --jitter 0.0075 --swa
# archive run A3 (train.py ghi de cung 1 path giua cac run)
import sys; sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH
os.makedirs('/kaggle/working/runs/A3', exist_ok=True)
for p in [MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH]:
    shutil.copy(p, '/kaggle/working/runs/A3/')
print('A3 archived')


## 9 — So sánh A1 vs A2 vs A3 → chọn theo VAL (tránh test leakage) + đóng gói

Bảng ablation cho `logs/GH-88/ablation.md`. Model ship = run thắng trên **val MAE**;
số test của cả 3 run đều ghi vào bảng (transparency), nhưng KHÔNG dùng test để chọn.


In [ ]:
import os, sys, glob, shutil, torch
os.chdir('/kaggle/working/ai-module'); sys.path.insert(0, '/kaggle/working/ai-module')
from scripts.train import evaluate, load_split
from src.core.config import INPUT_FEATURES, SPECTRAL_FEAT_DIM, D_MODEL, D_STATE, FEATURE_SCALER_VERSION
from src.models.soh_predictor import MambaSOHPredictor

Xv, Xfv, yv = load_split('data/processed/val.pt', FEATURE_SCALER_VERSION)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = {}
for run in ['A1', 'A2', 'A3']:
    ckpt = torch.load(glob.glob(f'/kaggle/working/runs/{run}/soh_mamba_*.pth')[0], map_location=device)
    m = MambaSOHPredictor(input_features=INPUT_FEATURES, feat_dim=SPECTRAL_FEAT_DIM, d_model=D_MODEL, d_state=D_STATE).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    val = evaluate(m, Xv, Xfv, yv, device)
    results[run] = {'val_mae': val['mae'], 'val_rmse': val['rmse'], 'test_mae': ckpt['test_mae'], 'test_rmse': ckpt['test_rmse']}
print(f"{'run':<4} {'val MAE':>8} {'val RMSE':>9} {'test MAE':>9} {'test RMSE':>10}")
for run, r in results.items():
    print(f"{run:<4} {r['val_mae']:>8.4f} {r['val_rmse']:>9.4f} {r['test_mae']:>9.4f} {r['test_rmse']:>10.4f}")
winner = min(results, key=lambda k: results[k]['val_mae'])
print('WINNER (by val MAE):', winner)

# dong goi: artifacts run thang + logs de tai ve commit
shutil.make_archive('/kaggle/working/gh88_artifacts', 'zip', f'/kaggle/working/runs/{winner}')
shutil.make_archive('/kaggle/working/gh88_all_runs', 'zip', '/kaggle/working/runs')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
for lg in logs[-3:]:
    shutil.copy(lg, '/kaggle/working/')
print('Tai ve: gh88_artifacts.zip (4 artifacts run thang) + gh88_all_runs.zip + 3 log (per-band MAE nam trong log)')


## 10 — Dọn output (xoá repo clone + runs/ — đã nằm trong zip)

Output tab chỉ còn 2 zip + 3 log → download nhanh, không bị treo vì hàng nghìn file repo.


In [ ]:
import os, shutil
os.chdir('/kaggle/working')  # roi khoi repo truoc khi xoa
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/runs', ignore_errors=True)  # da nam trong gh88_all_runs.zip
print('Output con lai:', sorted(os.listdir('/kaggle/working')))
